# Module 3 — Lab

The lesson measured six engines for you. This lab is the other way round: you take
the measurements, and the exercise each time is **reading** them.

**Every measurement here has a trap in it.** Four exercises, one trap each:

| | what you measure | the trap |
|---|---|---|
| 1 | a question of your own, on all six | a latency whose answer is wrong |
| 2 | the same Cypher text on three engines | "portable" is a claim, not a property |
| 3 | an algorithm no engine here has | the cost of leaving is not the wire |
| 4 | a write loop against a write batch | the engine is not what makes it slow |

**How to work.** Each exercise gives you the question, the tools, and the shape of
the answer. The query is yours to write. Each one ends in a number you can defend —
write the number down, and write down what it was measured against.

Budget: 1.5–2.5 hours.

**Before you start**, all four servers have to be up and the two embedded stores
loaded. The next cell checks, and tells you what is missing rather than failing on
the first query.

In [1]:
# The same `ask` the lesson uses: one call for six engines, rows of lists back.
# Two places this notebook can run from -- the module folder (the lab) and
# solutions/ one level down -- and the tools sit at the same place relative to
# the repository root from both.
import json, os, statistics, sys, time
from pathlib import Path

TOOLS = next(p for p in (Path("../../tools/03-where-data-lives"),
                         Path("../../../tools/03-where-data-lives")) if p.exists())
sys.path.insert(0, str(TOOLS))
from engines import ask, one, up, dialect, ENGINES, LPG, RDF, EMBEDDED, GRAPH, IMPORT
from queries import cypher_for, norm

EXPECTED = {**{e: 2_137_944 for e in LPG}, **{e: 36_840_344 for e in RDF}}
HOW = {"neo4j": "Bolt, port 7690", "falkordb": "Redis protocol, port 6379",
       "kuzu": "in-process, read-only", "graphdb": "SPARQL over HTTP, port 7200",
       "fuseki": "SPARQL over HTTP, port 3030", "oxigraph": "in-process, read-only"}

down = {e: why for e, why in up().items() if why}
print(f"{'engine':<11}{'holds':>14}   how you reach it")
for e in ENGINES:
    if e in down:
        print(f"{e:<11}{'DOWN':>14}   {down[e]}")
        continue
    n = int(one(e, "MATCH (n) RETURN count(n)") if e in LPG else
            one(e, "SELECT (COUNT(*) AS ?n) WHERE { GRAPH <%s> {?s ?p ?o} }" % GRAPH))
    flag = "" if n == EXPECTED[e] else f"   <-- expected {EXPECTED[e]:,}"
    print(f"{e:<11}{n:>14,}   {HOW[e]}{flag}")
print("""
Nodes on the three property graphs, triples on the three triple stores: the two
counts are not comparable with each other, and *The trip is not free* says why.
If an engine is flagged, stop and reload it -- the rest of this lab compares
engines, and that is only meaningful while they hold the same facts.""")

engine              holds   how you reach it
neo4j           2,137,944   Bolt, port 7690
falkordb        2,137,944   Redis protocol, port 6379
kuzu            2,137,944   in-process, read-only


graphdb        36,840,344   SPARQL over HTTP, port 7200


fuseki         36,840,344   SPARQL over HTTP, port 3030


oxigraph       36,840,344   in-process, read-only

Nodes on the three property graphs, triples on the three triple stores: the two
counts are not comparable with each other, and *The trip is not free* says why.
If an engine is flagged, stop and reload it -- the rest of this lab compares
engines, and that is only meaningful while they hold the same facts.


## 1. A question of your own, asked of all six

The lesson's ten questions were chosen to stress one thing each — an index, a
scan, an adjacency, a join. **Add an eleventh**, and make it one the lesson does
not already answer.

Pick something real that you would want to know from this data. For example: *which
licence type appears on the most licences held by companies that also hold city
contracts?* Or: *how many organisations share an address with another organisation
in a different OrganizationGroup?* Yours is better than either if it interests you.

Write it twice — Cypher and SPARQL — and ask all six.

**Produce:** a table with one row per engine, three columns: the answer, the median
of ten runs, and whether that answer equals the others. Then the two sentences that
matter: **which engine won, and why that is a property of your question rather than
of the engine.**

**Where to look.** *The same question, six times* does this for ten questions, and
its cell is the loop to copy. `tools/03-where-data-lives/queries.py` holds the ten
texts — a dict with `cypher`, `sparql` and a `stress` saying what the question
costs — and the `norm()` that makes two engines' answers comparable.

**The trap.** You will be tempted to write the query, read the times, and report
the winner. Do not. Check the answers first and expect a disagreement: the lesson's
answer check caught one, Q9, and it was the engines. The one you are likelier to
meet is a defect in the question.

That defect is a pattern that constrains less than the question does: a SPARQL
triple pattern with no `?x rdf:type voc:Label`, or a Cypher `MATCH (x)` with no
label. **Both languages allow it and neither warns** — on this data either one
answers 25,644 where the question wanted 2,957. It is not a property of one
paradigm, it is a question you did not finish writing.

**What the exercise really is:** deciding whether a gap you measured is the engine
or your query. One of those is worth reporting.

In [2]:
# 1. The licence type most common among companies that also hold city contracts.
#
# It has to cross the entity layer. The vendors on contracts and the holders of
# licences are different Organization rows -- the two source files never share
# one -- so "the same company" is only true one floor up, in OrganizationGroup.
# Every node in both texts names its class, which is the defect the trap is about.
MY_QUESTION = dict(
    id="Q11_licence_of_vendors",
    stress="five hops through the entity layer, then a group-by over licence types",
    cypher="""
        MATCH (:Contract)-[:HAS_VENDOR]->(:Organization)-[:BELONGS_TO_ORG_GROUP]->(g:OrganizationGroup)
              <-[:BELONGS_TO_ORG_GROUP]-(:Organization)-[:ORG_HAS_LICENSE]->(l:LicenseRecord)
              -[:HAS_LICENSE_TYPE]->(t:LicenseType)
        RETURN t.id AS type, count(DISTINCT l) AS licences
        ORDER BY licences DESC, type LIMIT 1""",
    sparql="""
        SELECT ?type (COUNT(DISTINCT ?l) AS ?licences) WHERE { GRAPH <%s> {
          ?c rdf:type voc:Contract ; voc:HAS_VENDOR ?o .
          ?o voc:BELONGS_TO_ORG_GROUP ?g . ?o2 voc:BELONGS_TO_ORG_GROUP ?g .
          ?o2 rdf:type voc:Organization ; voc:ORG_HAS_LICENSE ?l .
          ?l voc:HAS_LICENSE_TYPE ?t . ?t voc:id ?type } }
        GROUP BY ?type ORDER BY DESC(?licences) ?type LIMIT 1""" % GRAPH,
)
ITERATIONS, WARMUP = 10, 1

answers, medians = {}, {}
for e in [x for x in ENGINES if x not in down]:
    text = cypher_for(MY_QUESTION, e) if e in LPG else MY_QUESTION["sparql"]
    for _ in range(WARMUP):
        ask(e, text)
    ts = []
    for _ in range(ITERATIONS):
        t0 = time.perf_counter()
        rows = ask(e, text)
        ts.append((time.perf_counter() - t0) * 1000)
    answers[e], medians[e] = norm(rows), statistics.median(ts)

reference = answers[next(iter(answers))]
best = min(medians, key=medians.get)
print(f"{'engine':<11}{'answer':<26}{'median ms':>11}   same answer")
for e in answers:
    print(f"{e:<11}{str(answers[e][0]):<26}{medians[e]:>10,.0f}{'*' if e == best else ' '}"
          f"   {'yes' if answers[e] == reference else 'NO'}")
name = one("neo4j", "MATCH (t:LicenseType {id: '%s'}) RETURN t.description"
                    % reference[0][0].split(".")[0])
slowest = max(medians, key=medians.get)
print(f"""
licence type {reference[0][0].split('.')[0]} is "{name}", on {float(reference[0][1]):,.0f} licences.
{best} answered fastest; {slowest} took {medians[slowest] / medians[best]:,.0f} times as long.""")

engine     answer                      median ms   same answer
neo4j      ('1010.00', '15240.00')          784    yes
falkordb   ('1010.00', '15240.00')        1,616    yes
kuzu       ('1010.00', '15240.00')          110*   yes
graphdb    ('1010.00', '15240.00')        1,923    yes
fuseki     ('1010.00', '15240.00')        4,923    yes
oxigraph   ('1010.00', '15240.00')       10,283    yes

licence type 1010 is "Limited Business License", on 15,240 licences.
kuzu answered fastest; oxigraph took 94 times as long.


**All six agree, and that is checked before anything else:** licence type 1010,
*Limited Business License*, on 15,240 licences. No disagreement here, because every
node in both texts names its class — drop `?o2 rdf:type voc:Organization` or one
`:Organization` and the check is what would tell you.

**Kuzu won, and the reason is the shape of the question.** Five hops and a group-by
is a chain of joins over adjacency, which is exactly what Kuzu's columnar storage
and CSR adjacency are built for; the triple stores rebuild every hop as one more
self-join over the same quad indexes, so each hop the question adds multiplies
their bill. That is why the triple stores sit at the bottom of this row, and why
the same Oxigraph **won** Q1, a single lookup by URI, in the lesson.

So the sentence is not "Kuzu is fastest". It is: **a five-hop aggregation favours
the engine whose storage is an adjacency structure; a point lookup does not.**
Change the question and the winner changes — the lesson's thesis, in your own row.

## 2. The same Cypher text, on three engines that all speak Cypher

*The same question, six times* found one question where Neo4j and FalkorDB
answered 18,670 and Kuzu answered 18,898, on identical data and identical query
text. The cause was trail semantics against walk semantics in a variable-length
path: two engines refuse to reuse a relationship inside one path, one allows it.

**Find a second divergence of your own.** Variable-length paths are the obvious
place, but they are not the only one — look also at `shortestPath`, at `DISTINCT`
over a path, at ordering with ties, at how each engine prints a double, at what
each does with a null inside an aggregate, and at the type each loader gave a
value.

**Produce:** one query text, three answers, and an explanation that accounts for the
difference **numerically** — not "they implement it differently" but *this many
rows, for this reason, and here is the arithmetic*. The lesson's example closed
exactly: the gap was 228, which is the number of edges whose reverse also exists,
each one yielding one walk that repeats an edge.

**Where to look.** *The same question, six times*, the cell that closes Q9. Note how
the explanation is built: split the answer by path length, find which length
differs, then find a count in the data that equals the difference.

**The trap.** Kuzu stores `IS_SIMILAR_TO` and `BELONGS_TO_ORG_GROUP` as one table per
label pair, so the plain name does not exist there. `ask` rewrites the plain name
for the shapes the lesson uses — `dialect(text)` shows you exactly what Kuzu was
sent — and any other shape fails with `Binder exception: Table IS_SIMILAR_TO does
not exist`. Read that as the cost of the engine's DDL, not as a bug in your query,
and do not "fix" it by dropping the relationship type: an untyped pattern runs
everywhere and answers a different question.

**What the exercise really is:** producing an explanation that closes to the row.
An unexplained divergence is not a finding, it is a thing you noticed.

In [3]:
# 2. One text, three engines: a date comparison, which reads the same everywhere.
Q = "MATCH (r:ContractRecord) WHERE r.startDate > date('2020-01-01') RETURN count(r) AS n"
print("the text:", Q, "\n")
for e in LPG:
    print(f"   {e:<10}{int(one(e, Q)):>8,}")

# Split it. First: what IS a start date, on each engine? The driver says.
print("\nwhat comes back for one startDate:")
for e in LPG:
    v = one(e, "MATCH (r:ContractRecord) WHERE r.startDate IS NOT NULL "
               "RETURN r.startDate LIMIT 1")
    print(f"   {e:<10}{type(v).__module__ + '.' + type(v).__name__:<22}{v}")

# Then: is the comparison FALSE on FalkorDB, or not evaluated at all? A tautology
# answers that: `x OR NOT x` is true for every row where x is true or false, and
# drops only the rows where x is null.
TAUT = ("MATCH (r:ContractRecord) WHERE r.startDate IS NOT NULL AND "
        "(r.startDate > date('2020-01-01') OR NOT r.startDate > date('2020-01-01')) "
        "RETURN count(r)")
HAVE = "MATCH (r:ContractRecord) WHERE r.startDate IS NOT NULL RETURN count(r)"
STR = "MATCH (r:ContractRecord) WHERE r.startDate > '2020-01-01' RETURN count(r)"
print(f"\n{'':<13}{'have a date':>12}{'x OR NOT x':>12}{'> a STRING':>12}")
for e in LPG:
    print(f"   {e:<10}{int(one(e, HAVE)):>12,}{int(one(e, TAUT)):>12,}{int(one(e, STR)):>12,}")

the text: MATCH (r:ContractRecord) WHERE r.startDate > date('2020-01-01') RETURN count(r) AS n 

   neo4j       15,564


   falkordb         0
   kuzu        15,564

what comes back for one startDate:
   neo4j     neo4j.time.Date       1997-12-01
   falkordb  builtins.str          1997-12-01
   kuzu      datetime.date         1997-04-08

              have a date  x OR NOT x  > a STRING
   neo4j          121,076     121,076           0


   falkordb       121,076           0      15,564
   kuzu           121,076     121,076      15,564


**15,564 on Neo4j and Kuzu, 0 on FalkorDB — and the gap closes to the row.**

FalkorDB's start dates are **strings**: its bulk loader has no date type, so every
date column was declared `STRING` on the way in. Comparing a string with a `date()`
is not false there, it is **null** — the tautology `x OR NOT x` keeps all 121,076
dated records on Neo4j and Kuzu and **none** on FalkorDB. So the gap is not "some
rows": it is every one of the 15,564 records after 2020, because the comparison was
never evaluated on any row.

And the same text written with a string literal flips it: FalkorDB answers 15,564
— an ISO date compares correctly as text — while **Neo4j answers 0** for exactly the
same reason in the other direction. Kuzu answers 15,564 both ways because it casts
the string. Three engines, three rules, one query text, and no error from any of
them.

Two things to take away. The loader is part of the engine you are querying: this
store holds strings because of how it was loaded, not because FalkorDB lacks dates.
And `norm()` would have hidden it — `'1997-12-01'` and `date(1997, 12, 1)` print the
same — so an answer check that compares only printed values is not enough when a
**type** is what differs.

## 3. An algorithm none of the six has, and what leaving costs

*Where the computation lives* showed that the widest algorithm library available
is networkx, which is not a database, and that the toll for using it is flat: the
same rows leave the store whichever algorithm follows.

**Pick an algorithm no engine here offers** and run it on the organisation
similarity graph. `nx.articulation_points` is a good choice: nothing here has it,
and it answers a question worth asking — *which single company holds a cluster
together?* Betweenness on the whole graph is another: GDS has it, FalkorDB samples
it, and the lesson showed what that costs.

**Produce:** a table with four rows — the four ways out, Neo4j and the three triple
stores — each with `export`, `compute`, and the share of the total spent moving
rather than computing. Then the sentence that section hands over, in your own
numbers: **at what compute time does the export stop mattering?**

**Where to look.** *Where the computation lives*, under *The transfer, priced*: the
cell above that heading pulls the edge list out of all three triple stores.

**The trap, and it is the most useful thing in this lab.** An edge list cannot
carry a node that has no edges, and **189,238 of the 210,740 organisations have
none**. Pull the edges, build the graph, count the components, and you get 8,465
where the right answer is 197,703 — wrong by twenty-three times, entirely
plausible-looking, and nothing warns you. The isolated nodes have to be added
back by hand. This is not RDF's trap: it belongs to every path that takes data
out of a database, which includes every time you reach for pandas.

And one that is RDF's: the exporter percent-encodes the key in the URI, so
`contracts:10598723` arrives as `.../Organization/contracts%3A10598723`. Compare
that with an id from Neo4j and nothing matches — silently.

**What the exercise really is:** knowing what your export dropped. Measure it
both ways — with the isolated nodes and without — and report both numbers.

In [4]:
# 3. Articulation points: which single company holds a cluster together?
import urllib.parse
import networkx as nx

EDGES = {"cypher": "MATCH (a:Organization)-[:IS_SIMILAR_TO]->(b:Organization) RETURN a.id, b.id",
         "sparql": """SELECT ?a ?b WHERE { GRAPH <%s> {
             ?a voc:IS_SIMILAR_TO ?b . ?a rdf:type voc:Organization . ?b rdf:type voc:Organization
           } }""" % GRAPH}
NODES = {"cypher": "MATCH (o:Organization) RETURN o.id",
         "sparql": "SELECT ?o WHERE { GRAPH <%s> { ?o rdf:type voc:Organization } }" % GRAPH}
key = lambda uri: urllib.parse.unquote(uri.rsplit("/", 1)[-1])   # URI -> the LPG id

RESULT = {}
for e in [x for x in ("neo4j", "graphdb", "fuseki", "oxigraph") if x not in down]:
    lang = "cypher" if e in LPG else "sparql"
    t0 = time.perf_counter()
    edges = ask(e, EDGES[lang])
    nodes = ask(e, NODES[lang])
    export = time.perf_counter() - t0
    if lang == "sparql":
        edges = [(key(a), key(b)) for a, b in edges]
        nodes = [[key(n)] for n, in nodes]

    t0 = time.perf_counter()
    G = nx.Graph(edges)
    from_edges = nx.number_connected_components(G)
    cut = set(nx.articulation_points(G))
    G.add_nodes_from(n for n, in nodes)
    compute = time.perf_counter() - t0
    RESULT[e] = dict(export=export, compute=compute, from_edges=from_edges,
                     all=nx.number_connected_components(G), cut=cut, G=G)

print(f"{'way out':<11}{'export s':>9}{'compute s':>10}{'moving':>8}"
      f"{'components':>12}{'+ isolated':>12}{'cut nodes':>11}")
for e, r in RESULT.items():
    share = r["export"] / (r["export"] + r["compute"])
    print(f"{e:<11}{r['export']:>9.2f}{r['compute']:>10.2f}{share:>8.0%}"
          f"{r['from_edges']:>12,}{r['all']:>12,}{len(r['cut']):>11,}")
same = len({frozenset(r["cut"]) for r in RESULT.values()}) == 1
print(f"\nthe same articulation points from every way out: {'yes' if same else 'NO'}")

# The company that holds the biggest cluster together: the cut node whose
# removal leaves the largest piece of its component behind it.
G = RESULT["neo4j"]["G"]
giant = max(nx.connected_components(G), key=len)
H = G.subgraph(giant)
def stranded(v):
    pieces = nx.connected_components(H.subgraph(set(H) - {v}))
    return len(giant) - 1 - max(len(p) for p in pieces)
hub = max((v for v in RESULT["neo4j"]["cut"] if v in giant), key=stranded)
print(f"""largest cluster: {len(giant)} organisations. Remove {hub}
("{one('neo4j', "MATCH (o:Organization {id: '%s'}) RETURN o.name" % hub)}") and {stranded(hub)} of them lose
the main part of the cluster.""")

worst = max(r["export"] for r in RESULT.values())
print(f"""
For the export to be under a tenth of the bill, the algorithm has to cost nine
times the export: here about {9 * worst:.0f} s at the slowest way out. Articulation
points cost {RESULT['neo4j']['compute']:.2f} s; the exact betweenness on the whole graph took 97 s in
the lesson's record, which is where the export stops mattering.""")

way out     export s compute s  moving  components  + isolated  cut nodes
neo4j           2.26      0.12     95%       8,465     197,703      1,630
graphdb         1.53      0.09     94%       8,465     197,703      1,630
fuseki          1.96      0.09     95%       8,465     197,703      1,630
oxigraph        1.87      0.10     95%       8,465     197,703      1,630

the same articulation points from every way out: yes
largest cluster: 204 organisations. Remove licenses:431100
("A & B WIRELESS INC.") and 10 of them lose
the main part of the cluster.

For the export to be under a tenth of the bill, the algorithm has to cost nine
times the export: here about 20 s at the slowest way out. Articulation
points cost 0.12 s; the exact betweenness on the whole graph took 97 s in
the lesson's record, which is where the export stops mattering.


**8,465 components from the edges, 197,703 once the organisations are added back.**
The difference is the 189,238 organisations that have no similarity edge at all:
the edge list cannot carry them, so the first number is wrong by a factor of
twenty-three and looks perfectly reasonable. Articulation points do not move when
the isolated nodes return — a node with no edges cannot cut anything — which is
exactly why the trap is dangerous: an algorithm that happens to be immune to it
makes you trust the pipeline, and the next one is not.

**Moving dominates.** The computation is about a tenth of a second, so 94–97% of
every row is the export, and the four ways out disagree on how long the trip
takes, not on the answer: the same 1,630 cut nodes come back from all four. The
rule from the lesson holds in your own numbers — **the toll is flat and the
algorithm is not**. An algorithm has to cost nine times the export before the
export falls under a tenth of the bill; the exact betweenness on the whole graph,
at 97 s, is past that line, and articulation points are nowhere near it.

**And the question the algorithm was chosen for has a deflating answer.** 1,630
organisations are cut nodes, yet removing the most important one strands only
10 of the 204 in the largest cluster. No single company holds this graph
together. It is thousands of small stars, and the centre of a star is a cut node
by definition — which is why there are so many of them and why none matters
much. It is the shape the lesson's eigenvector zeros already pointed at, found
again by a different algorithm.

## 4. A write loop against a write batch

Every loader in this repository writes in batches, and the lesson read why from a
record: 2,000 rows written one at a time cost about 35 s on Fuseki and 0.02 s on
embedded Oxigraph — the same query language, over a thousand times apart.

**Reproduce it on two engines of your choice**, one served and one embedded, and
then project it.

**Produce:** for each engine, the time for N rows written singly, the time for the
same N in one statement, and the ratio. Then the projection: at that per-row rate,
**how long would the real load take row by row?** The payload is 4,073,458 edges.
Compare your projection with the batched load time in *The six engines*, which
`load-times.json` holds.

**Where to look.** The batching cell under *How the data got in* shows the two
shapes of statement. Write to a throwaway label or named graph — `BatchTest`,
`<http://kgtp.org/graph/batchtest>` — never to `kgtp`. The two embedded stores are
held read-only by this notebook, so for an embedded engine open a **new store in
a temporary directory**: same engine, same data structure on disk, and the loaded
one untouched. Fuseki takes SPARQL Update at `http://localhost:3030/ds/update`,
GraphDB at `http://localhost:7200/repositories/kgtp/statements`.

**The trap.** You will conclude that the wire is the cost. It is not, mostly.
In the lesson's record the two embedded engines come out at 3x and 14x, the two
served property graphs at 35x and 41x, and the two served triple stores at
hundreds — and GraphDB and Fuseki speak the **same protocol** as each other, on
the same machine, as Neo4j does over Bolt. What separates them is a durable
commit per statement, which a per-triple `INSERT DATA` pays every time.

**What the exercise really is:** separating two costs that arrive added together.
Your ratio has a round trip and a commit in it, and the only way to tell them
apart is to measure something that has one and not the other.

In [5]:
# 4. One at a time, then in one statement: three served engines, two embedded.
import tempfile, urllib.parse, urllib.request
import kuzu
import pyoxigraph as ox

TEST_GRAPH = "http://kgtp.org/graph/batchtest"
N = 2000
UPDATE = {"fuseki": "http://localhost:3030/ds/update",
          "graphdb": "http://localhost:7200/repositories/kgtp/statements"}
triple = lambda i: f'<http://kgtp.org/batchtest/{i}> <http://kgtp.org/batchtest/i> "{i}"'
insert = lambda body: f"INSERT DATA {{ GRAPH <{TEST_GRAPH}> {{ {body} }} }}"
drop = f"DROP SILENT GRAPH <{TEST_GRAPH}>"

def timed(fn):
    t0 = time.perf_counter(); fn(); return time.perf_counter() - t0

def served_rdf(engine):
    def send(text):
        urllib.request.urlopen(urllib.request.Request(
            UPDATE[engine], data=urllib.parse.urlencode({"update": text}).encode(),
            headers={"Content-Type": "application/x-www-form-urlencoded"}),
            timeout=600).read()
    send(drop)
    single = timed(lambda: [send(insert(triple(i))) for i in range(N)])
    send(drop)
    batch = timed(lambda: send(insert(" . ".join(triple(i) for i in range(N)))))
    send(drop)
    return single, batch

def neo4j():
    ask("neo4j", "MATCH (x:BatchTest) DETACH DELETE x")
    single = timed(lambda: [ask("neo4j", f"CREATE (:BatchTest {{i: {i}}})") for i in range(N)])
    ask("neo4j", "MATCH (x:BatchTest) DETACH DELETE x")
    rows = ", ".join(str(i) for i in range(N))      # the rows travel IN the statement
    batch = timed(lambda: ask("neo4j", f"UNWIND [{rows}] AS i CREATE (:BatchTest {{i: i}})"))
    ask("neo4j", "MATCH (x:BatchTest) DETACH DELETE x")
    return single, batch

def kuzu_tmp():
    db = kuzu.Database(os.path.join(tempfile.mkdtemp(), "batch.kuzu"))
    c = kuzu.Connection(db)
    c.execute("CREATE NODE TABLE BatchTest(i INT64, PRIMARY KEY(i))")
    single = timed(lambda: [c.execute(f"CREATE (:BatchTest {{i: {i}}})") for i in range(N)])
    c.execute("MATCH (x:BatchTest) DELETE x")
    rows = ", ".join(str(i) for i in range(N))
    batch = timed(lambda: c.execute(f"UNWIND [{rows}] AS i CREATE (:BatchTest {{i: i}})"))
    return single, batch

def oxigraph_tmp():
    store = ox.Store(tempfile.mkdtemp())             # on disk: the same RocksDB
    single = timed(lambda: [store.update(insert(triple(i))) for i in range(N)])
    store.update(drop)
    batch = timed(lambda: store.update(insert(" . ".join(triple(i) for i in range(N)))))
    return single, batch

RUN = {"graphdb": lambda: served_rdf("graphdb"), "fuseki": lambda: served_rdf("fuseki"),
       "neo4j": neo4j, "kuzu": kuzu_tmp, "oxigraph": oxigraph_tmp}
TIMES = json.load(open(os.path.join(IMPORT, "load-times.json")))
LOADED = {"neo4j": "neo4j-offline", "kuzu": "kuzu", "graphdb": "graphdb",
          "fuseki": "fuseki", "oxigraph": "oxigraph"}
EDGES = 4_073_458

print(f"{'':<10}{'singly s':>9}{'batch s':>9}{'ratio':>8}{'row by row':>13}{'batched load':>14}")
R = {}
for e, fn in RUN.items():
    if e in down and e not in EMBEDDED:
        continue
    single, batch = R[e] = fn()
    rec = TIMES[LOADED[e]]
    projected = single / N * EDGES
    shown = f"{projected / 3600:.1f} h" if projected >= 3600 else f"{projected / 60:.0f} min"
    print(f"{e:<10}{single:>9.2f}{batch:>9.3f}{single / batch:>7.0f}x"
          f"{shown:>13}{rec.get('total_s', rec['load_s']):>12.0f} s"
          + ("   (embedded)" if e in EMBEDDED else ""))

per_row = {e: s / N * 1000 for e, (s, b) in R.items()}
ms = lambda e: f"{per_row[e]:.2f} ms" if e in per_row else "not measured"
print(f"""
per row, written singly -- and what each one pays for:

   oxigraph   no wire, nothing durable per statement      {ms('oxigraph')}   the floor
   kuzu       no wire, a durable transaction per statement {ms('kuzu')}
   neo4j      a Bolt round trip and a transaction          {ms('neo4j')}
   graphdb    an HTTP round trip and a transaction         {ms('graphdb')}
   fuseki     an HTTP round trip and a transaction         {ms('fuseki')}""")
if "kuzu" in per_row and "oxigraph" in per_row:
    print(f"""
Kuzu and Oxigraph have the same wire -- none -- and are {per_row['kuzu'] / per_row['oxigraph']:,.0f}x apart per row.
There is no wire in that gap: what Kuzu pays per statement and Oxigraph does not
is a durable commit, and it is already the size of a served engine's bill.""")

           singly s  batch s   ratio   row by row  batched load


graphdb       13.72    0.052    263x        7.8 h          96 s


fuseki        55.37    0.097    568x       31.3 h         782 s


neo4j          3.12    0.093     34x        1.8 h          13 s


kuzu           8.20    0.064    128x        4.6 h          20 s   (embedded)
oxigraph       0.03    0.010      3x        1 min          45 s   (embedded)

per row, written singly -- and what each one pays for:

   oxigraph   no wire, nothing durable per statement      0.01 ms   the floor
   kuzu       no wire, a durable transaction per statement 4.10 ms
   neo4j      a Bolt round trip and a transaction          1.56 ms
   graphdb    an HTTP round trip and a transaction         6.86 ms
   fuseki     an HTTP round trip and a transaction         27.69 ms

Kuzu and Oxigraph have the same wire -- none -- and are 314x apart per row.
There is no wire in that gap: what Kuzu pays per statement and Oxigraph does not
is a durable commit, and it is already the size of a served engine's bill.


**The batch wins everywhere, and by very different factors** — which is the point.
The projection makes it concrete: written row by row, the 4,073,458 edges would
take hours on every served engine, and on Kuzu, against seconds to minutes for
the batched loads the lesson read from `load-times.json`. Only Oxigraph would
still finish in minutes.

**The two costs come apart on the two embedded engines, not on the wire.** Kuzu
and Oxigraph have no round trip at all, and they are hundreds of times apart per
row. What Kuzu pays and Oxigraph does not is a durable transaction per statement:
Oxigraph's update lands in the memory table of its LSM-tree and is made durable
later, in bulk. So a cost with **no wire in it** is already as large as a served
engine's whole per-row bill. Neo4j, GraphDB and Fuseki add a round trip on top
of their own transaction, and how much each costs is mostly how heavy that
transaction is — GraphDB and Fuseki both speak HTTP, and they are far apart.

This is not the lesson's record — there Kuzu came out at 14x, written into the
loaded store, and here at over a hundred, into a fresh one — and the difference is a
reason to report what your number was measured against, not to average the two.
What both agree on is the order: Oxigraph at the floor, every engine that commits
per statement far above it.

So "the network is slow" is the wrong lesson. **Batch because each statement is a
transaction, and only then because each statement is a trip.**

## Close

Four numbers, and for each one the thing it was measured against. If you wrote
down only the numbers, go back: a latency with no answer beside it, a ratio with no
control, and a component count with no note about what the export dropped are all
the same mistake in different clothes.

The one sentence to carry out of this lab: **a measurement you cannot say what it
was compared with is not a measurement.**